# Data Analyst Agent with OpenAI's Hosted Code Interpreter

This notebook builds an agent that writes and executes its own Python analysis code — not in this notebook's kernel, but in a **sandboxed container OpenAI hosts for you**, via the `code_interpreter` tool on the Responses API.

### Why a hosted sandbox?
* No local dependency wrangling for `pandas`/`seaborn`/`sklearn` inside the agent's execution environment — the sandbox already has a data-science stack installed.
* Untrusted, model-generated code never touches your own process.
* A container can be **reused across calls** to avoid a ~10-30s cold start on every request — we cache its ID and try to resume it first, falling back to a fresh container if it expired.

The task: load the classic Iris dataset, clean it, compute feature correlations, and save a Seaborn heatmap — all written and run by the model itself.

---
## Step 1: Install Dependencies

In [ ]:
!pip install -q openai python-dotenv

---
## Step 2: Set Up Environment Variables & Client

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv
from openai import BadRequestError, NotFoundError, OpenAI

load_dotenv()

if "OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"]:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

client = OpenAI()

---
## Step 3: Define the Task & Container-Reuse Helper
`CONTAINER_ID_FILE` persists the sandbox's container ID to disk (here, Colab's ephemeral local filesystem) so that re-running the notebook in the same session reuses the same warm container instead of paying the cold-start cost again.

In [ ]:
from pathlib import Path

CONTAINER_ID_FILE = Path(".container_id")
PROMPT = (
    "Using the code interpreter tool, execute Python code now to: load the iris dataset "
    "from sklearn.datasets, clean any missing data, compute the correlation between "
    "features, and save a Seaborn heatmap plot as iris_heatmap.png."
)


def run_in_sandbox(container):
    return client.responses.create(
        model="gpt-4o-mini",
        tools=[{"type": "code_interpreter", "container": container}],
        include=["code_interpreter_call.outputs"],
        input=PROMPT,
    )

---
## Step 4: Run the Agent
Try to resume a saved container first; if it expired (or none exists yet), start a fresh `"auto"` container. Either way, the model decides what code to write and calls the sandbox to run it.

In [ ]:
saved_container_id = CONTAINER_ID_FILE.read_text().strip() if CONTAINER_ID_FILE.exists() else None

if saved_container_id:
    print(f"Reusing container {saved_container_id}...")
    try:
        response = run_in_sandbox(saved_container_id)
    except (BadRequestError, NotFoundError):
        print("Saved container expired, starting a fresh one...")
        saved_container_id = None

if not saved_container_id:
    response = run_in_sandbox({"type": "auto"})

for item in response.output:
    if item.type == "code_interpreter_call":
        CONTAINER_ID_FILE.write_text(item.container_id)
        break

---
## Step 5: Retrieve the Generated Plot
The sandbox can hand back the saved file two different ways, so we check both: sometimes it's inlined as base64 directly on the `code_interpreter_call` output; other times it only shows up as a citation attached to the assistant's message, which requires a follow-up API call to fetch the file's bytes.

In [ ]:
import base64

saved = False

for item in response.output:
    if item.type == "message":
        for content in item.content:
            print(f"\nAssistant Response:\n{content.text}")
            # Fallback path: the sandbox sometimes reports the saved file only
            # as a citation on the message, not as inline output below.
            for annotation in getattr(content, "annotations", []):
                if getattr(annotation, "type", None) == "container_file_citation" and annotation.filename == "iris_heatmap.png":
                    image_bytes = client.containers.files.content.retrieve(
                        annotation.file_id, container_id=annotation.container_id
                    ).read()
                    with open("iris_heatmap.png", "wb") as f:
                        f.write(image_bytes)
                    saved = True
    elif item.type == "code_interpreter_call":
        # Fast path: the sandbox sometimes returns the image inline as base64.
        for output in item.outputs or []:
            if output.type == "image":
                image_bytes = base64.b64decode(output.url.split(",", 1)[1])
                with open("iris_heatmap.png", "wb") as f:
                    f.write(image_bytes)
                saved = True

print("Saved plot to iris_heatmap.png" if saved else "No plot file was produced.")

---
## Step 6: Display the Plot

In [ ]:
from IPython.display import Image, display

if saved:
    display(Image(filename="iris_heatmap.png"))